In [6]:
from bs4 import BeautifulSoup
import re
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils.rnn import pad_sequence
from torchtext.vocab import build_vocab_from_iterator
import nltk
from nltk.tokenize import word_tokenize
import os

In [5]:
regex = re.compile(r"\((\d+), '([^']+)', '([^']+)', (\d+), ('[^']+'\))")

mapper = dict()

count = 500

with open('n96ncsr5g4-1/index.txt', 'r') as file:
    fields = file.readlines()
    for field in fields:
        if count == 0:
            break
        count -= 1
        match = re.match(regex, field)
        if match:
            mapper[match.group(3)] = match.group(4)

print(mapper)


{'1613573972338075.html': '1', '1635698138155948.html': '0', '1635699228889266.html': '0', '1635750062162701.html': '0', '161356510250721.html': '0', '1635697720027445.html': '0', '1608573404300907.html': '1', '1635703553325743.html': '0', '160792972564439.html': '0', '1626171244686112.html': '0', '1626720628315328.html': '0', '1635714223550669.html': '0', '1635699761652684.html': '0', '1635704538042534.html': '0', '163570487542444.html': '0', '16249662296873.html': '1', '1613581349027434.html': '0', '1635706054338702.html': '0', '1624422656324031.html': '1', '1635703174277606.html': '0', '1613573052480813.html': '1', '1607095600394378.html': '1', '1620759901211522.html': '1', '1626464266508342.html': '1', '162314783573698.html': '0', '1635702757169414.html': '0', '1625542298164263.html': '1', '1635706754013304.html': '0', '1635707612573897.html': '0', '1635700926348099.html': '0', '1613555572947897.html': '0', '1613531087043553.html': '0', '1635701813408794.html': '0', '16357119085202

In [11]:
files = []

for part in range(1, 7):  # Dataset parts 1 to 6
    dataset_path = f'n96ncsr5g4-1/dataset/dataset-part-{part}'
    for file in os.listdir(dataset_path):
        full_path = f'{dataset_path}/{file}'
        files.append(full_path)

X = []
y = []
for file in files:
    # Extract the filename from the full path
    filename = file.split('/')[-1]
    if filename in mapper:
        with open(file, 'r') as file:
            html_content = file.read()
            soup = BeautifulSoup(html_content, 'html.parser')
            text = soup.get_text()
            text = text.strip()
            X.append(text)
            y.append(mapper[filename])

'A bowler to bank on | The Cricket Monthly | ESPN Cricinfo\n  \n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n          BROWSE BY SECTION\n         \n\n          Issues\n         \n\n\n\n\n\n\n\n           Analysis\n          \n\n\n\n           Comment\n          \n\n\n\n           Essay\n          \n\n\n\n           Features\n          \n\n\n\n           Hate to Love\n          \n\n\n\n           Interview\n          \n\n\n\n           Profile\n          \n\n\n\n           Photo Feature\n          \n\n\n\n           Talking Cricket\n          \n\n\n\n           Stats Feature\n          \n\n\n\n           Quiz\n          \n\n\n\n           High Fives\n          \n\n\n\n           Gleanings\n          \n\n\n\n           I Was There\n          \n\n\n\n\n\n\n\n\n\n\n\n\n\n\n         Sign in\n        \n\n         Logout\n        \n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\

# Preprocessing

In [14]:
nltk.download('punkt')

# Tokenization (character-level and word-level)
def tokenize_char(doc):
    return list(doc)  # Character-level tokenization

def tokenize_word(doc):
    return word_tokenize(doc)  # Word-level tokenization

char_sequences = [tokenize_char(_text) for _text in X]
word_sequences = [tokenize_word(_text) for _text in X] 

# Building the vocab from iterator
def yield_tokens(data_iter):
    for seq in data_iter:
        yield seq

char_vocab = build_vocab_from_iterator(yield_tokens(char_sequences), specials=["<unk>"])
word_vocab = build_vocab_from_iterator(yield_tokens(word_sequences), specials=["<unk>"])

# Setting default index for unknown tokens
char_vocab.set_default_index(char_vocab["<unk>"])
word_vocab.set_default_index(word_vocab["<unk>"])

# Check vocab
print("Character Vocabulary:", char_vocab.get_stoi())  # String-to-index mapping
print("Word Vocabulary:", word_vocab.get_stoi())  # String-to-index mapping


[nltk_data] Downloading package punkt to /Users/alfred/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


Character Vocabulary: {'🔖': 1442, '📦': 1441, '📌': 1440, '👋': 1437, '｜': 1436, '하': 1432, '터': 1430, '치': 1429, '용': 1425, '사': 1423, '는': 1421, '게': 1418, '龙': 1417, '黎': 1416, '顿': 1412, '類': 1411, '韩': 1408, '韦': 1407, '面': 1405, '陶': 1402, '陵': 1401, '降': 1400, '陀': 1399, '閉': 1396, '长': 1395, '链': 1392, '钟': 1391, '違': 1386, '道': 1385, '過': 1384, '週': 1381, '过': 1373, '越': 1372, '迎': 1374, '赞': 1370, '費': 1367, '豪': 1365, '銭': 1390, '误': 1362, '论': 1359, '融': 1353, '题': 1413, '蓬': 1352, '蒂': 1351, '葉': 1349, '菲': 1347, '麻': 1415, '莫': 1345, '航': 1341, '腊': 1339, '老': 1336, '继': 1333, '締': 1330, '線': 1329, '総': 1328, '結': 1325, '📄': 1439, '秘': 1318, '破': 1315, '真': 1314, '相': 1312, '益': 1310, '疑': 1308, '疆': 1307, '界': 1306, '班': 1303, '湾': 1297, '済': 1295, '洪': 1291, '泰': 1289, '아': 1424, '没': 1287, '汪': 1286, '汤': 1285, '気': 1282, '段': 1281, '欢': 1278, '検': 1277, '梵': 1275, '根': 1274, '柬': 1272, '暗': 1265, '旺': 1262, '旦': 1261, '族': 1260, '斐': 1259, '政': 1257, '掛': 1254, '挝': 1251

In [15]:
# Convert to numerical indices
char_indices = [[char_vocab[char] for char in seq] for seq in char_sequences]
word_indices = [[word_vocab[word] for word in seq] for seq in word_sequences]

# Convert to tensors and pad
char_indices = [torch.tensor(seq) for seq in char_indices]
word_indices = [torch.tensor(seq) for seq in word_indices]

char_padded = pad_sequence(char_indices, batch_first=True, padding_value=0)
word_padded = pad_sequence(word_indices, batch_first=True, padding_value=0)


In [ ]:
embedding_dim = 100

# Embedding layers
char_embedding = nn.Embedding(len(char_vocab), embedding_dim)
word_embedding = nn.Embedding(len(word_vocab), embedding_dim)

char_embedded = char_embedding(char_padded)
word_embedded = word_embedding(word_padded)

# Concatenate embeddings
concatenated = torch.cat((char_embedded, word_embedded), dim=1)
print(concatenated.shape)

In [ ]:
class HTMLPhishCNN(nn.Module):
    def __init__(self):
        super(HTMLPhishCNN, self).__init__()
        
        # 32 convolutional filters with 8 different kernel sizes
        self.conv_layers = nn.ModuleList([
            nn.Conv1d(in_channels=100, out_channels=32, kernel_size=k) for k in [3, 5, 7, 9, 11, 13, 15, 17]
        ])
        
        # Max Pooling layer
        self.pool = nn.MaxPool1d(kernel_size=2)
        
        # Placeholder for fully connected layer; dynamically set later
        self.fc = None
        
        # Output layer for binary classification
        self.output = nn.Linear(10, 1)
    
    def forward(self, x):
        # Apply Conv1D and ReLU to each layer
        x = [F.relu(conv(x.permute(0, 2, 1))) for conv in self.conv_layers]
        
        # Apply Max Pooling to each output
        x = [self.pool(conv) for conv in x]
        
        # Concatenate the output of all convolution layers along the sequence length
        x = torch.cat(x, dim=2)
        
        # Dynamically calculate the flattened size
        if self.fc is None:
            flatten_size = x.view(x.size(0), -1).size(1)
            self.fc = nn.Linear(flatten_size, 10)
        
        # Flatten the output
        x = x.view(x.size(0), -1)
        
        # Fully connected layer
        x = F.relu(self.fc(x))
        
        # Output layer (sigmoid for binary classification)
        x = torch.sigmoid(self.output(x))
        
        return x

# Example: create a model and pass the input
model = HTMLPhishCNN()

# Example input with shape [1, 4373, 100]
input_tensor = concatenated
output = model(input_tensor)
output # Should output the prediction for binary classification


In [ ]:
# Define loss function and optimizer
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Assuming we have labels for our data
# For this example, let's create dummy labels
labels = torch.randint(0, 2, (input_tensor.size(0), 1)).float()

# Number of epochs
num_epochs = 10

# Training loop
for epoch in range(num_epochs):
    # Forward pass
    outputs = model(input_tensor)
    
    # Compute loss
    loss = criterion(outputs, labels)
    
    # Backward pass and optimize
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    # Print progress
    if (epoch + 1) % 2 == 0:
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')

# Test the model
model.eval()
with torch.no_grad():
    test_output = model(input_tensor)
    predicted = (test_output > 0.5).float()
    accuracy = (predicted == labels).float().mean()
    print(f'Test Accuracy: {accuracy.item():.4f}')

# Don't forget to set the model back to training mode if you plan to train further
model.train()
